#### 서면질문서 회신용 자료추출
1. 지역 상권 매출 및 유동 인구 변화 관련

* 1-1. 최근 3년간(2023년~2025년) 사직야구장 프로야구 홈 경기일과 비경기일(동일 요일 기준)의 사직야구장 반경 500m 이내(초근접 상권)와 
*      반경 500m~1.5km 이내(골목상권 및 전통시장 구역)의 시간대별 신용카드 결제액 및 결제 건수(외식업, 숙박업, 도소매업 등 업종별 분류 포함)
* 예시) 부산시 보유 민간 신용카드사 가맹점 매출 빅데이터 및 동백전 결제 데이터 등

In [1]:
import pandas as pd
import glob
import warnings
warnings.filterwarnings('ignore')

In [ ]:
area1 = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\대상영역\\area1_g100m.csv", dtype={'id': 'str'}, encoding="utf-8")
area2 = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\대상영역\\area2_g100m.csv", dtype={'id': 'str'}, encoding="utf-8")
seccode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\bccd_sec_code.csv", sep="|", encoding="utf-8")
seccode = seccode[['ry_m_cd', 'ry_m_nm']]
seccode = seccode.drop_duplicates(keep='first',subset='ry_m_cd')
tme_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\02_카드이용\\*_tme_*.csv", recursive=True)
sec_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\02_카드이용\\*_sec_*.csv", recursive=True)
playinfo = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\sajik_playstatus.csv")
output_path = r"D:\\04_자료제출\\2026년\\20260709서면질문서\\결과자료\\"

# 사직야구장 반경 조건과 경기일정을 기준을 카드이용 현황 추출
def extract1(data_lists, area):
    res_lists = []    
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str}, encoding="utf-8")
        df = pd.merge(area, df, on="id", how="left")
        df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]').astype(str)
        df = pd.merge(playinfo, df, left_on='날짜', right_on='std_ymd')
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)                   
    return concated

# 사직야구장 비경기일 기준을 카드이용 현황 추출
def extract2(data_lists, area):
    res_lists = []    
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str}, encoding="utf-8")
        df = pd.merge(area, df, on="id", how="left")
        df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]')
        df['weekname'] = df['std_ymd'].dt.day_name('ko_KR')
        df['std_ymd'] = df['std_ymd'].astype('str')
        df = pd.merge(df, playinfo[['날짜','구장']], left_on="std_ymd", right_on="날짜", how='outer')
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)                   
    return concated

def aggregate1(data_lists, area):
    df = extract1(data_lists, area)
    if 'tme' in df.columns:
        df['tme'] = df['tme'].astype('str')
        df['시간'] = df['tme'] + "시"
        df.loc[df['tme'].str.contains('00|01|02|03'), '시간대'] = '심야'
        df.loc[df['tme'].str.contains('04|05|06|07'), '시간대'] = '새벽'
        df.loc[df['tme'].str.contains('08|09|10|11'), '시간대'] = '오전'
        df.loc[df['tme'].str.contains('12|13|14|15'), '시간대'] = '오후'
        df.loc[df['tme'].str.contains('16|17|18|19'), '시간대'] = '저녁'
        df.loc[df['tme'].str.contains('20|21|22|23'), '시간대'] = '야간'
        group = df.groupby(['날짜','요일','승패','시간','시간대'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'})
        group.columns = ['날짜','요일','승패','시간','시간대','이용금액','이용건수']
    elif 'ry_m_cd' in df.columns:
        df = pd.merge(df, seccode, on='ry_m_cd', how='inner')
        df = df[~df['ry_m_cd'].str.contains('M007|M008|M028|M029|M030|M031|M032')]
        df.rename(columns={'ry_m_nm': '업종중분류'}, inplace=True)
        group = df.groupby(['날짜','요일','승패','업종중분류'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'})
        group.columns = ['날짜','요일','승패','업종중분류','이용금액','이용건수']
    return group

def aggregate2(data_lists, area):
    df = extract2(data_lists, area)
    df = df[df['날짜'].isna()].copy()
    if 'tme' in df.columns:
        df['tme'] = df['tme'].astype('str')
        df['시간'] = df['tme'] + "시"
        df.loc[df['tme'].str.contains('00|01|02|03'), '시간대'] = '심야'
        df.loc[df['tme'].str.contains('04|05|06|07'), '시간대'] = '새벽'
        df.loc[df['tme'].str.contains('08|09|10|11'), '시간대'] = '오전'
        df.loc[df['tme'].str.contains('12|13|14|15'), '시간대'] = '오후'
        df.loc[df['tme'].str.contains('16|17|18|19'), '시간대'] = '저녁'
        df.loc[df['tme'].str.contains('20|21|22|23'), '시간대'] = '야간'
        group = df.groupby(['std_ymd','weekname','시간','시간대'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'})
        group.columns = ['날짜','요일','시간','시간대','이용금액','이용건수']
    elif 'ry_m_cd' in df.columns:
        df = pd.merge(df, seccode, on='ry_m_cd', how='inner')
        df = df[~df['ry_m_cd'].str.contains('M007|M008|M028|M029|M030|M031|M032')]
        df.rename(columns={'ry_m_nm': '업종중분류'}, inplace=True)
        group = df.groupby(['std_ymd','weekname','업종중분류'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'})
        group.columns = ['날짜','요일','업종중분류','이용금액','이용건수']
    return group

# aggregate1(tme_lists, area1).to_excel(output_path + "1-1_경기일_초근접상권_시간대 카드이용 현황.xlsx", engine="openpyxl", index=False)
# aggregate1(tme_lists, area2).to_excel(output_path + "1-1_경기일_골목상권_시간대 카드이용 현황.xlsx", engine="openpyxl", index=False)
# aggregate1(sec_lists, area1).to_excel(output_path + "1-1_경기일_초근접상권_업종별 카드이용 현황.xlsx", engine="openpyxl", index=False)
# aggregate1(sec_lists, area2).to_excel(output_path + "1-1_경기일_골목상권_업종별 카드이용 현황.xlsx", engine="openpyxl", index=False)

aggregate2(tme_lists, area1).to_excel(output_path + "1-1_비경기일_초근접상권_시간대 카드이용 현황3.xlsx", engine="openpyxl", index=False)
aggregate2(tme_lists, area2).to_excel(output_path + "1-1_비경기일_골목상권_시간대 카드이용 현황3.xlsx", engine="openpyxl", index=False)
aggregate2(sec_lists, area1).to_excel(output_path + "1-1_비경기일_초근접상권_업종별 카드이용 현황3.xlsx", engine="openpyxl", index=False)
aggregate2(sec_lists, area2).to_excel(output_path + "1-1_비경기일_골목상권_업종별 카드이용 현황3.xlsx", engine="openpyxl", index=False)


* 1-2. 통신사 기지국 데이터 기반 경기 시작 3시간 전 및 종료 후 3시간 동안의 동래구(사직동·온천동) 유입 인구 동선, 체류 시간, 연령대별 자료
* 예시) '부산시 생활인구 및 모바일(SKT/KT) 통신 위치기반 데이터

In [ ]:
import pandas as pd
import glob

jscode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\행정경계코드\\KIKcd_H20260301_include_oldcode.csv", encoding="utf-8")

inflow = jscode[~jscode['행정동코드'].astype(str).str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000')]
inflow = inflow[(~inflow['시도명'].str.contains('직할시|출장소|부산')) & (inflow['시군구명'].isna()) & (inflow['읍면동명'].isna())].copy()
inflow = inflow[['행정동코드', '시도명']]
inflow['inflow_cd'] = inflow['행정동코드'].astype(str).str[:2] + "00000000"
inflow.rename(columns={'시도명': 'inflow_nm'}, inplace=True)
inflow = inflow[['inflow_cd','inflow_nm']]

hjdong = pd.DataFrame({
    'hcode': ['2626055000','2626056000','2626057000','2626058000','2626059000','2626060000'],
    'hname': ['온천제1동','온천제2동','온천제3동','사직제1동','사직제2동','사직제3동']})
emd_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\01_생활인구\\03_유입인구\\*_emd_*.csv", recursive=True)
tme_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\01_생활인구\\03_유입인구\\*_tme_*.csv", recursive=True)
playinfo = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\sajik_playstatus.csv")
output_path = r"D:\\04_자료제출\\2026년\\20260709서면질문서\\결과자료\\"

# 데이터 전처리 함수
def extract(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str}, encoding="utf-8")
        cond1 = (df['hcode'].str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000'))
        cond2 = (~df['inflow_cd'].str.startswith('26'))
        # cond3 = (~df['inflow_cd'].str.startswith('99'))
        df = df[cond1 & cond2].copy()
        df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]').astype(str)
        df = pd.merge(playinfo[['날짜','요일','시작시간']], df, left_on="날짜", right_on="std_ymd", how="inner")
        df['inflow_cd'] = df['inflow_cd'].str[:2] + '00000000'
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)
    return concated

# 전처리 후 집계
def aggregate_emd(data_lists):
    df = extract(data_lists)
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    df = pd.merge(df, inflow, on="inflow_cd", how="inner")
    df['sum_pop'] = df.loc[:, 'h_pop':'v_pop'].sum(axis=1)
    df.loc[df['gen_age'].str.contains('0009|1014|1519'), 'ages'] = '20대미만'
    df.loc[df['gen_age'].str.contains('2024|2529'), 'ages'] = '20대'
    df.loc[df['gen_age'].str.contains('3034|3539'), 'ages'] = '30대'
    df.loc[df['gen_age'].str.contains('4044|4549'), 'ages'] = '40대'
    df.loc[df['gen_age'].str.contains('5054|5559'), 'ages'] = '50대'
    df.loc[df['gen_age'].str.contains('6064|6569'), 'ages'] = '60대'
    df.loc[df['gen_age'].str.contains('7074|7500'), 'ages'] = '70대이상'
    group = df.groupby(['날짜','요일','hname','ages','inflow_nm'], as_index=False)['sum_pop'].sum().round().reset_index()
    group = group[['날짜','요일','hname','ages','inflow_nm','sum_pop']]
    group.columns = ['날짜','요일','행정동명','연령','유입지명','유입인구수']
    return group

# 시작시간 3시간 후 유입인구
def aggregate_tme1(data_lists):
    df = extract(data_lists)
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    df = pd.merge(df, inflow, on="inflow_cd", how="inner")
    df['sum_pop'] = df.loc[:, 'h_pop':'v_pop'].sum(axis=1)
    df['시간'] = df['tme'].astype(str) + "시"
    df['시간차'] = [int(df.loc[i, '시작시간'].split(':')[0]) - int(df.loc[i, 'tme']) for i in df.index]
    df = df[df['시간차'] == 3].copy()
    group = df.groupby(['날짜','요일','시작시간','hname','시간','시간차','inflow_nm'], as_index=False)['sum_pop'].sum().round().reset_index()
    group = group[['날짜','요일','시작시간','hname','시간','시간차','inflow_nm','sum_pop']]
    group.columns = ['날짜','요일','시작시간','행정동명','시간','시간차','유입지명','유입인구수']
    return group

# 종료시간 3시간 후 유입인구 
# 평균 경기시간이 3시간이므로 시작시간에서 3시간을 더하면 종료시간임.
# 종료시간에 3시간을 더하면 18시 30분 경기는 다음 날 00시 30분을 의미함.
# 경기일이든 비경기일이든 00시대 유입인구보다 23시대 유입인구로 제공
def aggregate_tme2(data_lists):
    df = extract(data_lists)
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    df = pd.merge(df, inflow, on="inflow_cd", how="inner")
    df['sum_pop'] = df.loc[:, 'h_pop':'v_pop'].sum(axis=1)
    df['시간'] = df['tme'].astype(str) + "시"
    df['시간차'] = [int(df.loc[i, '시작시간'].split(':')[0]) + 5 for i in df.index]
    df = df[df['시간차'].astype(str) == df['tme'].astype(str)].copy()
    group = df.groupby(['날짜','요일','시작시간','hname','시간','시간차','inflow_nm'], as_index=False)['sum_pop'].sum().round().reset_index()
    group = group[['날짜','요일','시작시간','hname','시간','시간차','inflow_nm','sum_pop']]
    group.columns = ['날짜','요일','시작시간','행정동명','시간','시간차','유입지명','유입인구수']
    return group

aggregate_emd(emd_lists).to_excel(output_path + "1-2_경기일_연령대_유입인구_현황.xlsx", engine="openpyxl", index=False)
aggregate_tme1(tme_lists).to_excel(output_path + "1-2_경기일_시작시간3시간전_시간대_유입인구_현황.xlsx", engine="openpyxl", index=False)
aggregate_tme2(tme_lists).to_excel(output_path + "1-2_경기일_종료시간2시간후_시간대_유입인구_현황.xlsx", engine="openpyxl", index=False)


* 경기일 vs 비경기일 부산 타구군, 타시도 유입인구의 카드이용 현황

In [11]:
import pandas as pd
import glob

jscode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\행정경계코드\\KIKcd_H20260301_include_oldcode.csv", encoding="utf-8")

# 부산 타 구군 코드
ssgcd = jscode[(jscode['행정동코드'].astype(str).str.startswith('26'))]
ssgcd = ssgcd[(ssgcd['시군구명'] != '동래구') & (~ssgcd['시군구명'].isna())].copy()
ssgcd = ssgcd[['행정동코드', '시군구명']]
ssgcd['cln_ssg_cd'] = ssgcd['행정동코드'].astype(str).str[:5]
ssgcd.rename(columns={'시군구명': 'cln_ssg_nm'}, inplace=True)
ssgcd = ssgcd[['cln_ssg_cd','cln_ssg_nm']]
ssgcd['cln_ssg_cd'] = ssgcd['cln_ssg_cd'].astype(str)
ssgcd = ssgcd.drop_duplicates(keep='first')

# 부산 외 타 시도코드
ctycd = jscode[~jscode['행정동코드'].astype(str).str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000')]
ctycd = ctycd[(~ctycd['시도명'].str.contains('직할시|출장소|부산')) & (ctycd['시군구명'].isna()) & (ctycd['읍면동명'].isna())].copy()
ctycd = ctycd[['행정동코드', '시도명']]
ctycd['cln_cty_cd'] = ctycd['행정동코드'].astype(str).str[:2]
ctycd.rename(columns={'시도명': 'cln_cty_nm'}, inplace=True)
ctycd = ctycd[['cln_cty_cd','cln_cty_nm']]
ctycd['cln_cty_cd'] = ctycd['cln_cty_cd'].astype(str)

crd_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\02_카드이용\\*_emd_*.csv", recursive=True)
playinfo = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\sajik_playstatus.csv")
output_path = r"D:\\04_자료제출\\2026년\\20260709서면질문서\\제출자료\\"
hjdong = pd.DataFrame({
    'hcode': ['2626055000','2626056000','2626057000','2626058000','2626059000','2626060000'],
    'hname': ['온천제1동','온천제2동','온천제3동','사직제1동','사직제2동','사직제3동']})

# 데이터 전처리 함수
def extract_ssg(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'cln_cty_cd': str, 'cln_ssg_cd': str}, encoding="utf-8")
        cond1 = (df['hcode'].str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000'))
        cond2 = (df['cln_cty_cd'] == '26')
        df = df[cond1 & cond2].copy()
        df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]').astype(str)
        df = pd.merge(playinfo[['날짜','요일','시작시간']], df, left_on="날짜", right_on="std_ymd", how="outer")
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)
    return concated

def extract_cty(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'cln_cty_cd': str, 'cln_ssg_cd': str}, encoding="utf-8")
        cond1 = (df['hcode'].str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000'))
        cond2 = (df['cln_cty_cd'] != '26')
        df = df[cond1 & cond2].copy()
        df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]').astype(str)
        df = pd.merge(playinfo[['날짜','요일','시작시간']], df, left_on="날짜", right_on="std_ymd", how="outer")
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)
    return concated

# 전처리 후 집계
# def aggregate_ssg1(data_lists):
#     df = extract_ssg(data_lists)
#     df = pd.merge(df, hjdong, on="hcode", how="inner")
#     df = pd.merge(df, ssgcd, on="cln_ssg_cd", how="inner")
#     group = df.groupby(['날짜','요일','hname','cln_ssg_nm'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'}).round().reset_index()
#     group = group[['날짜','요일','hname','cln_ssg_nm','amt','cnt']]
#     group.columns = ['날짜','요일','행정동명','구군명','이용금액','이용건수']
#     return group

# 전처리 후 집계
# def aggregate_cty1(data_lists):
#     df = extract_cty(data_lists)
#     df = pd.merge(df, hjdong, on="hcode", how="inner")
#     df = pd.merge(df, ctycd, on="cln_cty_cd", how="inner")
#     group = df.groupby(['날짜','요일','hname','cln_cty_nm'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'}).round().reset_index()
#     group = group[['날짜','요일','hname','cln_cty_nm','amt','cnt']]
#     group.columns = ['날짜','요일','행정동명','시도명','이용금액','이용건수']
#     return group

# 전처리 후 집계
def aggregate_ssg2(data_lists):
    df = extract_ssg(data_lists)
    df = df[df['날짜'].isna()].copy()
    df['weekname'] = df['std_ymd'].astype('datetime64[ns]').dt.day_name('ko_KR')
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    df = pd.merge(df, ssgcd, on="cln_ssg_cd", how="inner")
    group = df.groupby(['std_ymd','weekname','hname','cln_ssg_nm'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'}).round().reset_index()
    group = group[['std_ymd','weekname','hname','cln_ssg_nm','amt','cnt']]
    group.columns = ['비경기일','요일','행정동명','구군명','이용금액','이용건수']
    return group

# 전처리 후 집계
def aggregate_cty2(data_lists):
    df = extract_cty(data_lists)
    df = df[df['날짜'].isna()].copy()
    df['weekname'] = df['std_ymd'].astype('datetime64[ns]').dt.day_name('ko_KR')
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    df = pd.merge(df, ctycd, on="cln_cty_cd", how="inner")
    group = df.groupby(['std_ymd','weekname','hname','cln_cty_nm'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'}).round().reset_index()
    group = group[['std_ymd','weekname','hname','cln_cty_nm','amt','cnt']]
    group.columns = ['날짜','요일','행정동명','시도명','이용금액','이용건수']
    return group

# aggregate_ssg1(crd_lists).to_excel(output_path + "1-2_경기일_타구군 카드이용_현황.xlsx", engine="openpyxl", index=False)
# aggregate_cty1(crd_lists).to_excel(output_path + "1-2_경기일_타시도 카드이용_현황.xlsx", engine="openpyxl", index=False)

# aggregate_ssg2(crd_lists).to_excel(output_path + "1-2_비경기일_타구군 카드이용_현황.xlsx", engine="openpyxl", index=False)
aggregate_cty2(crd_lists).to_excel(output_path + "1-2_비경기일_타시도 카드이용_현황.xlsx", engine="openpyxl", index=False)



* 1-3. 소상공인시장진흥공단 상권정보시스템 기준, 프로야구 시즌(4월~10월)과 비시즌(11월~3월) 동안 사직동·온천동 일대 골목상권 점포의 폐업률 및 평균 존속기간 변동 현황을 확인할 수 있는 자료
* 예시) 가맹점 주소지 좌표 기반 신용카드사 매출 증감 분석 지표
* 예시) 소상공인시장진흥공단(SEMAS) 상권정보시스템 지표 및 지방행정 인허가 휴·폐업 신고 대장

In [53]:
# 경기일 기준으로 지방행정 인허가 데이터를 병합
# 업태 중 병원, 의원, 백화점은 제외

data_lists = glob.glob(r"부산광역시_지방행정인허가\\*.csv", recursive=True)
playinfo = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\sajik_playstatus.csv")

results = []
for data in data_lists:
    df = pd.read_csv(data, encoding="cp949", low_memory=False)
    df = df[(df['인허가일자'].str[:4].astype(int) >= 2023) & (df['인허가일자'].str[:4].astype(int) <= 2025)].copy()
    for i in df.index:
        if df.loc[i, '도로명주소'] == '':
            df.loc[i, '도로명주소'] =  df.loc[i, '지번주소']
            df = df[['인허가일자','영업상태명','업태구분명','폐업일자','휴업시작일자','휴업종료일자','도로명주소']]
    results.append(df)
concated_df = pd.concat(results, ignore_index=True)
concated_df = concated_df[(concated_df['도로명주소'].str.contains('동래구')) & (concated_df['도로명주소'].str.contains('사직동|온천동'))]

# 시즌기간기준 인허가 현황 추출
merged_df1 = pd.merge(playinfo, concated_df, left_on="날짜", right_on="인허가일자", how="inner")
merged_df1 = merged_df1[(merged_df1['업태구분명'] != '병원') & (merged_df1['업태구분명'] != '의원') & (merged_df1['업태구분명'] != '백화점')]
merged_df1 = merged_df1[['날짜','요일','승패','인허가일자','영업상태명','폐업일자','휴업시작일자','휴업종료일자','도로명주소']]

# 시즌기간 골목상권 점포의 폐업률 -> 시즌기간 영업상태별 건수와 비율 산출
merged_df1['시즌기간'] = merged_df1['날짜'].str[:7]
count1 = merged_df1[['시즌기간','영업상태명']].value_counts()
ratio1 = merged_df1[['시즌기간','영업상태명']].value_counts(normalize=True).round(3)
result1 = pd.merge(count1, ratio1, on=['시즌기간','영업상태명'], how="inner").reset_index()
result1.columns = ['시즌기간','영업상태명','현황','비율']
# result1.to_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\결과자료\\1-3_시즌기간_골목상권 점포_영업현황.xlsx", engine='openpyxl', index=False)

# 시즌기간 평균 존속기간 변동 현황
result2 = merged_df1[['날짜','영업상태명','인허가일자','폐업일자']]
# result1.rename(columns={'날짜': '시즌기간'}, inplace=True)
result2 = result2[result2['영업상태명'].str.contains('폐업|제외')]
result2['인허가일자'] = result2['인허가일자'].astype('datetime64[ns]')
result2['폐업일자'] = result2['폐업일자'].astype('datetime64[ns]')
result2['존속기간'] = (result2['폐업일자'] - result2['인허가일자']).dropna()
result2['존속기간'] = result2['존속기간'].dt.days
result2['시즌기간'] = result2['날짜'].str[:7]
result2_g = result2.groupby('시즌기간', as_index=False)['존속기간'].mean().round()
# result1_g.to_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\결과자료\\1-3_시즌기간_골목상권 존속기간 현황.xlsx", engine='openpyxl', index=False)

# 비시즌기간 인허가 현황 추출
merged_df2 = pd.merge(concated_df, playinfo, left_on="인허가일자", right_on="날짜", how="left")
merged_df2 = merged_df2[merged_df2['날짜'].isna()]
merged_df2['비시즌기간'] = merged_df1['인허가일자'].str[:7]
count2 = merged_df2[['비시즌기간','영업상태명']].value_counts()
ratio2 = merged_df2[['비시즌기간','영업상태명']].value_counts(normalize=True).round(3)
result3 = pd.merge(count2, ratio2, on=['비시즌기간','영업상태명'], how="inner").reset_index()
result3.columns = ['비시즌기간','영업상태명','현황','비율']
result3.to_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\결과자료\\1-3_비시즌기간_골목상권 점포_영업현황.xlsx", engine='openpyxl', index=False)

# 비시즌기간 평균 존속기간 변동 현황
result4 = pd.merge(concated_df, playinfo, left_on="인허가일자", right_on="날짜", how="left")
result4 = result4[result4['날짜'].isna()]
result4 = result4[result4['영업상태명'].str.contains('폐업|제외')]
result4['인허가일자'] = result4['인허가일자'].astype('datetime64[ns]')
result4['폐업일자'] = result4['폐업일자'].astype('datetime64[ns]')
result4['존속기간'] = (result4['폐업일자'] - result4['인허가일자']).dropna()
result4['존속기간'] = result4['존속기간'].dt.days
result4['비시즌기간'] = result4['인허가일자'].astype(str).str[:7]
result4_g = result4.groupby('비시즌기간', as_index=False)['존속기간'].mean().round()
result4_g.to_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\결과자료\\1-3_비시즌기간_골목상권 존속기간 현황.xlsx", engine='openpyxl', index=False)


2. 외부 유입 인구의 지역경제 기여도 관련
* 2-1. 최근 3년간 주말 경기일 기준, 동래구 외 지역(부산 타 구·군 및 타 시·도)에 주소지를 둔 카드 소지자가 사직동·온천동 상권에서 소비한 금액 및 전체 매출 중 점유율 현황
* 예시) 한국관광 데이터랩(Data Lab) 및 카드사 거주지 기반 소비데이터

In [63]:

jscode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\행정경계코드\\KIKcd_H20260301_include_oldcode.csv", encoding="utf-8")

# 부산 타 구군 코드
inflow1 = jscode[(jscode['행정동코드'].astype(str).str.startswith('26'))]
inflow1 = inflow1[(inflow1['시군구명'] != '동래구') & (~inflow1['시군구명'].isna())].copy()
inflow1 = inflow1[['행정동코드', '시군구명']]
inflow1['cln_ssg_cd'] = inflow1['행정동코드'].astype(str).str[:5]
inflow1.rename(columns={'시군구명': 'cln_ssg_nm'}, inplace=True)
inflow1 = inflow1[['cln_ssg_cd','cln_ssg_nm']]

# 부산 외 타 시도코드
inflow2 = jscode[~jscode['행정동코드'].astype(str).str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000')]
inflow2 = inflow2[(~inflow2['시도명'].str.contains('직할시|출장소|부산')) & (inflow2['시군구명'].isna()) & (inflow2['읍면동명'].isna())].copy()
inflow2 = inflow2[['행정동코드', '시도명']]
inflow2['cln_cty_cd'] = inflow2['행정동코드'].astype(str).str[:2]
inflow2.rename(columns={'시도명': 'cln_cty_nm'}, inplace=True)
inflow2 = inflow2[['cln_cty_cd','cln_cty_nm']]

hjdong = pd.DataFrame({
    'hcode': ['2626055000','2626056000','2626057000','2626058000','2626059000','2626060000'],
    'hname': ['온천제1동','온천제2동','온천제3동','사직제1동','사직제2동','사직제3동']})
emd_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\02_카드이용\\*_emd_*.csv", recursive=True)
playinfo = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\sajik_playstatus.csv")
output_path = r"D:\\04_자료제출\\2026년\\20260709서면질문서\\결과자료\\"

# 데이터 전처리(부산 타구군) 함수
def extract1(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'cln_ssg_cd': str}, encoding="utf-8")
        cond1 = (df['hcode'].str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000'))
        cond2 = ((df['cln_ssg_cd'].str.startswith('26')) & (df['cln_ssg_cd'] != '26260'))
        df = df[cond1 & cond2].copy()
        df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]').astype(str)
        df = pd.merge(playinfo[['날짜','요일','시작시간']], df, left_on="날짜", right_on="std_ymd", how="inner")
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)
    concated = concated[concated['요일'].str.contains('토|일')].copy()
    return concated

# 데이터 전처리(타 시도) 함수
def extract2(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'cln_cty_cd': str}, encoding="utf-8")
        cond1 = (df['hcode'].str.contains('2626055000|2626056000|2626057000|2626058000|2626059000|2626060000'))
        cond2 = (~df['cln_cty_cd'].str.startswith('26'))
        df = df[cond1 & cond2].copy()
        df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]').astype(str)
        df = pd.merge(playinfo[['날짜','요일','시작시간']], df, left_on="날짜", right_on="std_ymd", how="inner")
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)
    concated = concated[concated['요일'].str.contains('토|일')].copy()
    return concated

# 전처리 후 집계
def aggregate1(data_lists):
    df = extract1(data_lists)
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    df = pd.merge(df, inflow1, on="cln_ssg_cd", how="inner")
    group = df.groupby(['날짜','요일','hname','cln_ssg_nm'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'}).round().reset_index()
    group = group[['날짜','요일','hname','cln_ssg_nm','amt','cnt']]
    group.columns = ['날짜','요일','행정동명','시군구명','이용금액','이용건수']
    return group

# 전처리 후 집계
def aggregate2(data_lists):
    df = extract2(data_lists)
    df = pd.merge(df, hjdong, on="hcode", how="inner")
    df = pd.merge(df, inflow2, on="cln_cty_cd", how="inner")
    group = df.groupby(['날짜','요일','hname','cln_cty_nm'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'}).round().reset_index()
    group = group[['날짜','요일','hname','cln_cty_nm','amt','cnt']]
    group.columns = ['날짜','요일','행정동명','유입지명','이용금액','이용건수']
    return group

# 연도별 동래구로 유입된 카드이용 전체금액 집계
def summary(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str}, encoding="utf-8")
        df = df[df['hcode'].str.startswith('26260')].copy()
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)
    concated['연도'] = concated['std_ymd'].astype('datetime64[ns]').astype(str).str[:4] + "년"
    group = concated.groupby('연도', as_index=False).agg({'amt': 'sum', 'cnt': 'sum'})
    group.columns = ['연도','이용금액','이용건수']
    return group


# aggregate1(emd_lists).to_excel(output_path + "2-1_주말경기일기준_부산 타구군거주자_카드이용 현황.xlsx", engine="openpyxl", index=False)
# aggregate2(emd_lists).to_excel(output_path + "2-1_주말경기일기준_타시도거주자_카드이용 현황.xlsx", engine="openpyxl", index=False)
summary(emd_lists).to_excel(output_path + "2-1_최근 3년간 연도별 카드이용 현황(합산).xlsx", engine="openpyxl", index=False)


In [ ]:
# 기업신용 데이터에서 2023년~2025년 동래구 소개 기업매출액 추출
import pandas as pd

pb_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\05_기업신용\\*_pb_*.csv", recursive=True) # 사업장정보
fs_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\05_기업신용\\*_fs_*.csv", recursive=True) # 매출매입정보

# 데이터 리스트를 데이터 프레임으로 변경하기 위한 빈 리스트 생성
res_lists1 = []
res_lists2 = []
for pb in pb_lists:
    df = pd.read_csv(pb, sep="|", dtype={'kedcd': str}, encoding="utf-8")
    df = df[(df['loc_addr'].str.contains('부산 동래구')) & (~df['loc_addr'].isna())].copy()
    res_lists1.append(df)
pb_df = pd.concat(res_lists1, ignore_index=True)

for fs in fs_lists:
    res_lists2.append(pd.read_csv(fs, sep="|", dtype={'kedcd': str}, encoding="utf-8"))
fs_df = pd.concat(res_lists2, ignore_index=True)

merged = pd.merge(pb_df[['kedcd','ksic10_bzc_nm','lqdt_cd']], fs_df[['kedcd','acct_dt','val']], on="kedcd", how="left")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9359224 entries, 0 to 9359223
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   kedcd          object 
 1   ksic10_bzc_nm  object 
 2   lqdt_cd        object 
 3   acct_dt        float64
 4   val            float64
dtypes: float64(2), object(3)
memory usage: 357.0+ MB


* 2-2. 사직야구장 경기 개최 주간의 동래구 방문객 수 및 관광 소비 지출액 변화 추이
* 예시) 한국관광 데이터랩(Data Lab) 지자체 권한 접속을 통한 사직동 일대 지출 세부 지표

In [31]:
# 사직야구장 경기 개최주간 동래구 방문객 -> 사직야구장 방문객
# 통신데이터의 방문인구 -> 동래구에 주거하지도, 직장을 다니지도 않는 인구
# 관광소비 업종은 한국관광데이터랩 업종을 참고하여 선별함
# 레저, 기타관광, 쇼핑몰, 기타숙박, 호텔, 캠핑/펜션, 제과음료, 일반외식, 기타레저, 골프장, 문화서비스, 관광유원시설, 여행업, 육상운송, 렌터카, 뷰티
# BC카드 업종중분류: M001, M004, M005, M007, M008, M014, M015, M016, M017, M018, M021, M026

# 방문객 수
tme_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\01_생활인구\\03_유입인구\\*_tme_*.csv", recursive=True)
crd_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\02_카드이용\\*_emd_*.csv", recursive=True)
playinfo = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\sajik_playstatus.csv", encoding="utf-8")
seccode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\bccd_sec_code.csv", sep="|", encoding="utf-8")
seccode = seccode[['ry_m_cd', 'ry_m_nm']]
seccode = seccode.drop_duplicates(keep='first',subset='ry_m_cd')
output_path = r"D:\\04_자료제출\\2026년\\20260709서면질문서\\"

# 요일 컬럼을 이용해 날짜가 속한 주의 목요일을 구하고, 그 목요일이 속한 연도/월/주차를 부여
# (목요일이 처음 등장하는 주가 그 달의 1주차 -> 월이 바뀌어도 그 주 목요일이 새 달이면 새 달 1주차)
dow_map = {'월': 0, '화': 1, '수': 2, '목': 3, '금': 4, '토': 5, '일': 6}
playinfo['날짜'] = playinfo['날짜'].astype('datetime64[ns]')
offset = 3 - playinfo['요일'].map(dow_map)
thursday = playinfo['날짜'] + pd.to_timedelta(offset, unit='D')
playinfo['연도'] = thursday.dt.strftime('%Y') + "년"
playinfo['월'] = thursday.dt.strftime('%m') + "월"
playinfo['주차'] = ((thursday.dt.day - 1) // 7 + 1).astype(str) + "주차"
playinfo['날짜'] = playinfo['날짜'].astype(str)

def aggregate_vpop(data_lists, df2):
    res_lists = []
    for data in data_lists:
        df1 = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str}, encoding="utf-8")
        df1 = df1[df1['hcode'].str.startswith('26260')].copy()
        df1['std_ymd'] = df1['std_ymd'].astype('datetime64[ns]').astype(str)
        res_lists.append(pd.merge(df2[['날짜','연도','월','주차']], df1[['std_ymd','v_pop']], left_on='날짜', right_on='std_ymd', how="inner"))
    concated = pd.concat(res_lists, ignore_index=True)
    group = concated.groupby(['연도','월','주차'], as_index=False)['v_pop'].sum().round()
    group.columns = ['연도','월','주차','방문인구']
    return group

def aggregate_card(data_lists, df2):
    res_lists = []
    for data in data_lists:
        df1 = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str}, encoding="utf-8")
        df1['std_ymd'] = df1['std_ymd'].astype('datetime64[ns]').astype(str)
        cond1 = (df1['hcode'].str.startswith('26260'))
        cond2 = (df1['ry_m_cd'].str.contains('M001|M004|M005|M007|M008|M014|M015|M016|M017|M018|M021|M026'))
        df1 = df1[cond1 & cond2].copy()
        df1 = pd.merge(df1, seccode, on="ry_m_cd", how="inner")
        res_lists.append(pd.merge(df2[['날짜','연도','월','주차']], df1[['std_ymd','ry_m_nm','amt','cnt']],
                                  left_on="날짜", right_on="std_ymd", how="inner"))
    concated = pd.concat(res_lists, ignore_index=True)
    group = concated.groupby(['연도','월','주차','ry_m_nm'], as_index=False).agg({'amt': 'sum', 'cnt': 'sum'})
    group.columns = ['연도','월','주차','업종중분류','이용금액','이용건수']
    return group        

# aggregate_vpop(tme_lists, playinfo).to_excel(output_path + "2-2_경기개최주간 동래구 방문인구 현황.xlsx", engine="openpyxl", index=False)
aggregate_card(crd_lists, playinfo).to_excel(output_path + "2-2_경기개최주간 동래구 관광소비 현황.xlsx", engine="openpyxl", index=False)


#### 사직야구장건립팀 요청사항
* 시즌(4월~10월)/비시즌(11월~3월) 초근접상권(500m 이내), 골목상권-전통시장(500m~1.5km) 요일구간별 영업일수, 총매출액, 일평균 매출액, 결제건수, 건당 결제액
* 시즌: 2023년~2025년, 비시즌: 2026년 2월까지
* 영업일수는 경기일-카드기준연월일 연동하여 산출

In [53]:
import pandas as pd
import numpy as np
import glob
pd.options.display.float_format = '{:.0f}'.format

playinfo = pd.read_csv(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\sajik_playstatus.csv", encoding="utf-8")
playinfo['시즌기간'] = playinfo['날짜'].str[:7]
playcnt = playinfo.groupby('시즌기간', as_index=False)['날짜'].count().reset_index()
playcnt.rename(columns={'날짜': '영업일수'}, inplace=True)
nonplay = pd.read_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\추가요청\\1-1_비경기일_초근접상권_시간대 카드이용 현황4.xlsx", engine='openpyxl')
nonplay = nonplay[['날짜','요일']]
nonplay.drop_duplicates(keep='first', inplace=True, ignore_index=True)
nonplay['비시즌기간'] = nonplay['날짜'].str[:7]
nonplaycnt = nonplay.groupby('비시즌기간', as_index=False)['날짜'].count()
nonplaycnt.rename(columns = {'날짜': '영업일수'}, inplace=True)
season_lists = glob.glob(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\추가요청\\*_경기일_*.xlsx", recursive=True)
nonseason_lists = glob.glob(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\추가요청\\*_비경기일_*.xlsx", recursive=True)

def aggregate_season(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_excel(data, engine='openpyxl')
        df['시즌기간'] = df['날짜'].str[:7]
        df.loc[df['요일'].str.contains('화|수|목'), '요일구분'] = '화~목요일'
        df.loc[df['요일'].str.contains('금|토|일'), '요일구분'] = '금~일요일'
        if '초근접상권' in data:
            df['상권구분'] = '초근접상권(500m 이내)'
        elif '골목상권' in data:
            df['상권구분'] = '골목상권·전통시장(500m~1.5km 이내)'
        
        group = df.groupby(['시즌기간','상권구분','요일구분'], as_index=False)[['이용금액','이용건수']].sum()
        group = pd.merge(group, playcnt, on='시즌기간', how="left")
        res_lists.append(group)
    concated = pd.concat(res_lists)
    concated = concated[['시즌기간','상권구분','요일구분','영업일수','이용금액','이용건수']]
    concated.rename(columns={'이용금액': '총매출액', '이용건수': '결제건수'}, inplace=True)
    concated['일평균매출액'] = (concated['총매출액'] / concated['영업일수']).round()
    concated['건당 결제액'] = (concated['총매출액'] / concated['결제건수']).round()
    concated = concated[['시즌기간','상권구분','요일구분','영업일수','총매출액','일평균매출액','결제건수','건당 결제액']]
    return concated

def aggregate_nonseason(data_lists):
    res_lists = []
    for data in data_lists:
        df = pd.read_excel(data, engine='openpyxl')
        df['비시즌기간'] = df['날짜'].str[:7]
        df.loc[df['요일'].str.contains('화|수|목'), '요일구분'] = '화~목요일'
        df.loc[df['요일'].str.contains('금|토|일'), '요일구분'] = '금~일요일'
        if '초근접상권' in data:
            df['상권구분'] = '초근접상권(500m 이내)'
        elif '골목상권' in data:
            df['상권구분'] = '골목상권·전통시장(500m~1.5km 이내)'
        
        group = df.groupby(['비시즌기간','상권구분','요일구분'], as_index=False)[['이용금액','이용건수']].sum()
        group = pd.merge(group, nonplaycnt, on='비시즌기간', how='left')
        res_lists.append(group)
    concated = pd.concat(res_lists)
    concated = concated[['비시즌기간','상권구분','요일구분','영업일수','이용금액','이용건수']]
    concated.rename(columns={'이용금액': '총매출액', '이용건수': '결제건수'}, inplace=True)
    concated['일평균매출액'] = (concated['총매출액'] / concated['영업일수']).round()
    concated['건당 결제액'] = (concated['총매출액'] / concated['결제건수']).round()
    concated = concated[['비시즌기간','상권구분','요일구분','영업일수','총매출액','일평균매출액','결제건수','건당 결제액']]
    return concated

aggregate_season(season_lists).to_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\추가요청\\야구시즌_상권구분_요일구분_매출현황.xlsx", engine='openpyxl', index=False)
aggregate_nonseason(nonseason_lists).to_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\추가요청\\비시즌_상권구분_요일구분_매출현황.xlsx", engine='openpyxl', index=False)


In [52]:
nonplay1 = pd.read_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\추가요청\\1-1_비경기일_초근접상권_시간대 카드이용 현황4.xlsx", engine='openpyxl')
nonplay1['비시즌기간'] = nonplay1['날짜'].str[:7]
nonplaycnt1 = nonplay1.groupby('비시즌기간', as_index=False)['날짜'].count()
nonplaycnt1.rename(columns = {'날짜': '영업일수'}, inplace=True)


nonplay2 = pd.read_excel(r"D:\\04_자료제출\\2026년\\20260709서면질문서\\추가요청\\1-1_비경기일_골목상권_시간대 카드이용 현황4.xlsx", engine='openpyxl')
nonplay2 = nonplay2[['날짜','요일']]
nonplay2.drop_duplicates(keep='first', inplace=True, ignore_index=True)
nonplay2['비시즌기간'] = nonplay2['날짜'].str[:7]
nonplaycnt2 = nonplay2.groupby('비시즌기간', as_index=False)['날짜'].count()
nonplaycnt2.rename(columns = {'날짜': '영업일수'}, inplace=True)
nonplaycnt2

# pd.merge(nonplaycnt1, nonplaycnt2, on='비시즌기간', how='inner')

,비시즌기간,영업일수
0,2023-03,21
1,2023-04,15
2,2023-05,19
3,2023-06,18
4,2023-07,24
5,2023-08,16
6,2023-09,20
7,2023-10,25
8,2023-11,30
9,2023-12,31
